# Model Training
Train baseline supervised models using the processed dataset. This notebook avoids exploratory analysis and focuses on preparing and fitting models.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
CLASSIFICATION_RESULTS_FILE = "classification_model_results.csv"
REGRESSION_RESULTS_FILE = "regression_model_results.csv"


In [2]:
processed_data = pd.read_csv("../data/processed/cleaned_data.csv")
target_classification = processed_data["Unemployability_Risk"]
target_regression = processed_data["Employment_Rate_12_Months (%)"]
feature_columns = [col for col in processed_data.columns if col not in ["Unemployability_Risk", "Employment_Rate_12_Months (%)"]]
X = processed_data[feature_columns]
X.head()


,Degree_Level,Graduation_Year,Skill_1,Skill_2,Skill_3,Skill_Demand_Score (1–100),Remote_Work_Availability (%),Employer_Reputation_Score (1–100),Degree_Level_Ordinal,Graduation_Recency,Demand_x_Reputation,Demand_x_Remote,Reputation_x_Remote,Field_of_Study_Computer Science,Field_of_Study_Data Science & AI,Field_of_Study_Engineering,Field_of_Study_Healthcare & Medicine,Field_of_Study_Natural Sciences,Field_of_Study_Social Sciences
0,Bachelor,2017,0,16,17,0.259208,8.8,66,1,2,4554,607.2,580.8,0,0,1,0,0,0
1,Bachelor,2023,16,0,17,0.395184,65.4,63,1,8,4473,4643.4,4120.2,0,0,1,0,0,0
2,Master,2019,2,10,22,-0.896584,5.0,74,2,4,3848,260.0,370.0,0,0,0,1,0,0
3,Master,2016,9,25,3,0.259208,10.3,48,2,1,3312,710.7,494.4,1,0,0,0,0,0
4,PhD,2023,13,6,19,0.259208,64.0,65,3,8,4485,4416.0,4160.0,0,0,0,0,0,0


In [3]:
def build_preprocessor(feature_df):
    categorical_columns = feature_df.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
    numeric_columns = [col for col in feature_df.columns if col not in categorical_columns]
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot)
    ])
    return ColumnTransformer([
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns)
    ])

def train_classification_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    models = {
        "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced", random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
    }
    results = []
    preprocessor = build_preprocessor(X_train)
    for name, estimator in models.items():
        pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        probabilities = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else pipeline.decision_function(X_test)
        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(y_test, predictions, zero_division=0),
            "Recall": recall_score(y_test, predictions, zero_division=0),
            "F1": f1_score(y_test, predictions, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, probabilities)
        })
    return pd.DataFrame(results)

def train_regression_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    models = {
        "Ridge Regression": Ridge(alpha=1.0),
        "Random Forest Regressor": RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE),
        "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=RANDOM_STATE)
    }
    results = []
    preprocessor = build_preprocessor(X_train)
    for name, estimator in models.items():
        pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        results.append({
            "Model": name,
            "R2": r2_score(y_test, predictions),
            "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
            "MAE": mean_absolute_error(y_test, predictions)
        })
    return pd.DataFrame(results)


In [4]:
classification_results = train_classification_models(X, target_classification)
regression_results = train_regression_models(X, target_regression)
classification_results.to_csv(CLASSIFICATION_RESULTS_FILE, index=False)
regression_results.to_csv(REGRESSION_RESULTS_FILE, index=False)
print("Classification results saved to", CLASSIFICATION_RESULTS_FILE)
print("Regression results saved to", REGRESSION_RESULTS_FILE)
classification_results
regression_results


Classification results saved to classification_model_results.csv
Regression results saved to regression_model_results.csv


,Model,R2,RMSE,MAE
0,Ridge Regression,0.459343,4.637800,3.670545
1,Random Forest Regressor,0.412400,4.834951,3.848457
2,Gradient Boosting Regressor,0.447690,4.687516,3.720324
